<a href="https://colab.research.google.com/github/alexmcostantini/MA-Scanner/blob/main/Weekly_Options_Scanner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import datetime as dt
import warnings
import numpy as np
import pandas as pd
from scipy.stats import norm
import yfinance as yf

warnings.filterwarnings("ignore")

# ==============================================================================
# 1. CONFIGURATION & SCREENING THRESHOLDS
# ==============================================================================
WATCHLIST = [
    "SPY", "QQQ", "IWM",          # Bucket A: Broad Index ETFs
    "XLF", "XLE", "GDX", "IBIT",  # Bucket B: Sector & Crypto ETFs
    "AAPL", "AMD", "NVDA", "PLTR",# Bucket C: Liquid Mega/Large-Cap Tech
    "SOFI", "F", "SNAP", "AAL"    # Bucket D: Sub-$15 Wheel / Cash-Secured Puts
]

RISK_FREE_RATE = 0.042        # Annualized risk-free rate (4.2%)
MIN_DTE = 4                   # Minimum Days to Expiration
MAX_DTE = 14                  # Maximum Days to Expiration (Weekly window)
MAX_COLLATERAL = 1000.0       # Hard cap per position ($1,000)
TARGET_SPREAD_WIDTHS = [2.5, 5.0]  # Preferred vertical spread widths ($)

# Contract Liquidity Filters
MIN_STRIKE_OI = 500           # Minimum Open Interest at specific strike
MIN_STRIKE_VOL = 100          # Minimum Daily Volume at specific strike
MAX_BID_ASK_SPREAD_LOW = 0.08 # Max bid-ask spread for options < $2.00
MAX_BID_ASK_SPREAD_HIGH = 0.15# Max bid-ask spread for options >= $2.00


# ==============================================================================
# 2. BLACK-SCHOLES DELTA & VOLATILITY RANK HELPERS
# ==============================================================================
def calc_bs_delta(S, K, T, r, sigma, option_type="call"):
    """Calculates Black-Scholes Delta for an option array or scalar."""
    if T <= 0 or sigma <= 0:
        return 0.0
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    if option_type == "call":
        return norm.cdf(d1)
    else:
        return norm.cdf(d1) - 1.0  # Put delta returned as negative (-1.0 to 0.0)


def get_spot_and_hv_bounds(ticker_obj):
    """Fetches current spot price and 52-week rolling 20-day HV min/max to compute IV Rank."""
    hist = ticker_obj.history(period="1y")
    if hist.empty or len(hist) < 40:
        return None, None, None
    spot = float(hist["Close"].iloc[-1])
    log_returns = np.log(hist["Close"] / hist["Close"].shift(1))
    rolling_hv = log_returns.rolling(window=20).std() * np.sqrt(252)
    hv_min = float(rolling_hv.min())
    hv_max = float(rolling_hv.max())
    return spot, hv_min, hv_max


def select_weekly_expiration(expirations, min_dte=MIN_DTE, max_dte=MAX_DTE):
    """Finds the best weekly expiration between min_dte and max_dte."""
    today = dt.date.today()
    valid_exps = []
    for exp_str in expirations:
        exp_date = dt.datetime.strptime(exp_str, "%Y-%m-%d").date()
        dte = (exp_date - today).days
        if min_dte <= dte <= max_dte:
            valid_exps.append((exp_str, dte))
    # Prefer the expiration closest to 7-10 DTE
    if not valid_exps:
        return None, None
    valid_exps.sort(key=lambda x: abs(x[1] - 8))
    return valid_exps[0]


def filter_liquid_chain(df, spot, T, option_type="put"):
    """Applies bid-ask, volume, open interest, and Delta calculations to an option chain."""
    df = df.copy()
    df["volume"] = df["volume"].fillna(0)
    df["openInterest"] = df["openInterest"].fillna(0)
    df["mid"] = (df["bid"] + df["ask"]) / 2.0
    df["ba_spread"] = df["ask"] - df["bid"]

    # Filter out zero-bid or overly wide bid-ask spreads
    max_allowed_spread = np.where(
        df["mid"] < 2.0, MAX_BID_ASK_SPREAD_LOW, MAX_BID_ASK_SPREAD_HIGH
    )
    liquid_mask = (
        (df["bid"] > 0.01)
        & (df["ba_spread"] <= max_allowed_spread)
        & (df["openInterest"] >= MIN_STRIKE_OI)
        & (df["volume"] >= MIN_STRIKE_VOL)
        & (df["impliedVolatility"] > 0.01)
    )
    df = df[liquid_mask].copy()
    if df.empty:
        return df

    # Compute Black-Scholes absolute Delta
    df["abs_delta"] = df.apply(
        lambda row: abs(
            calc_bs_delta(
                spot,
                row["strike"],
                T,
                RISK_FREE_RATE,
                row["impliedVolatility"],
                option_type,
            )
        ),
        axis=1,
    )
    return df


# ==============================================================================
# 3. STRATEGY BUILDERS (CREDIT SPREAD, DEBIT SPREAD, CASH-SECURED PUT)
# ==============================================================================
def find_bull_put_credit_spread(puts_df):
    """Finds a Bull Put Credit Spread (0.20-0.30 Delta short put, $2.50 or $5.00 wide)."""
    short_candidates = puts_df[
        (puts_df["abs_delta"] >= 0.20) & (puts_df["abs_delta"] <= 0.30)
    ]
    if short_candidates.empty:
        return None

    # Pick short leg closest to 0.25 Delta
    short_leg = short_candidates.iloc[
        (short_candidates["abs_delta"] - 0.25).abs().argsort()[:1]
    ].iloc[0]

    for width in TARGET_SPREAD_WIDTHS:
        target_long_strike = short_leg["strike"] - width
        long_matches = puts_df[np.isclose(puts_df["strike"], target_long_strike)]
        if not long_matches.empty:
            long_leg = long_matches.iloc[0]
            # Conservative pricing: sell at mid, buy at mid
            net_credit = round(short_leg["mid"] - long_leg["mid"], 2)
            collateral = round((width - net_credit) * 100, 2)
            credit_to_width = net_credit / width

            # Require >= 25% of width in credit and < $1,000 collateral
            if credit_to_width >= 0.25 and 0 < collateral <= MAX_COLLATERAL:
                ror = round((net_credit * 100 / collateral) * 100, 1)
                return {
                    "Strategy": "Bull Put Credit Spread",
                    "Setup": f"Sell ${short_leg['strike']:g}P / Buy ${long_leg['strike']:g}P",
                    "Deltas": f"{short_leg['abs_delta']:.2f} / {long_leg['abs_delta']:.2f}",
                    "Credit/Debit": f"+${net_credit * 100:.0f} Credit",
                    "Collateral/Cost": f"${collateral:.0f}",
                    "Max Return %": f"{ror}%",
                }
    return None


def find_bull_call_debit_spread(calls_df):
    """Finds a Bull Call Debit Spread (0.55-0.65 Delta long call, $2.50 or $5.00 wide)."""
    long_candidates = calls_df[
        (calls_df["abs_delta"] >= 0.55) & (calls_df["abs_delta"] <= 0.65)
    ]
    if long_candidates.empty:
        return None

    long_leg = long_candidates.iloc[
        (long_candidates["abs_delta"] - 0.60).abs().argsort()[:1]
    ].iloc[0]

    for width in TARGET_SPREAD_WIDTHS:
        target_short_strike = long_leg["strike"] + width
        short_matches = calls_df[np.isclose(calls_df["strike"], target_short_strike)]
        if not short_matches.empty:
            short_leg = short_matches.iloc[0]
            net_debit = round(long_leg["mid"] - short_leg["mid"], 2)
            total_cost = round(net_debit * 100, 2)
            debit_to_width = net_debit / width

            # Require debit <= 50% of width and total cost <= $1,000
            if 0.15 <= debit_to_width <= 0.50 and total_cost <= MAX_COLLATERAL:
                max_profit = (width - net_debit) * 100
                ror = round((max_profit / total_cost) * 100, 1)
                return {
                    "Strategy": "Bull Call Debit Spread",
                    "Setup": f"Buy ${long_leg['strike']:g}C / Sell ${short_leg['strike']:g}C",
                    "Deltas": f"{long_leg['abs_delta']:.2f} / {short_leg['abs_delta']:.2f}",
                    "Credit/Debit": f"-${total_cost:.0f} Debit",
                    "Collateral/Cost": f"${total_cost:.0f}",
                    "Max Return %": f"{ror}%",
                }
    return None


def find_cash_secured_put(puts_df, spot):
    """Finds a sub-$1,000 Cash-Secured Put on stocks priced under $12."""
    if spot > 12.0:
        return None
    short_candidates = puts_df[
        (puts_df["abs_delta"] >= 0.22)
        & (puts_df["abs_delta"] <= 0.32)
        & (puts_df["strike"] * 100 <= MAX_COLLATERAL)
    ]
    if short_candidates.empty:
        return None

    short_leg = short_candidates.iloc[
        (short_candidates["abs_delta"] - 0.27).abs().argsort()[:1]
    ].iloc[0]
    credit = round(short_leg["mid"] * 100, 2)
    collateral = round(short_leg["strike"] * 100 - credit, 2)
    weekly_yield = round((credit / (short_leg["strike"] * 100)) * 100, 2)

    if weekly_yield >= 1.0:
        return {
            "Strategy": "Cash-Secured Put (Wheel)",
            "Setup": f"Sell 1x ${short_leg['strike']:g}P",
            "Deltas": f"{short_leg['abs_delta']:.2f}",
            "Credit/Debit": f"+${credit:.0f} Credit",
            "Collateral/Cost": f"${collateral:.0f}",
            "Max Return %": f"{weekly_yield}% / wk",
        }
    return None


# ==============================================================================
# 4. MAIN SCANNER EXECUTION
# ==============================================================================
def run_weekly_scanner():
    results = []
    print(f"Scanning {len(WATCHLIST)} tickers for weekly setups under ${MAX_COLLATERAL:,.0f}...\n")

    for symbol in WATCHLIST:
        try:
            tk = yf.Ticker(symbol)
            spot, hv_min, hv_max = get_spot_and_hv_bounds(tk)
            if spot is None or not tk.options:
                continue

            exp_str, dte = select_weekly_expiration(tk.options)
            if not exp_str:
                continue

            chain = tk.option_chain(exp_str)
            T = dte / 365.0

            # Compute ATM Implied Volatility & IV Rank Proxy
            calls_raw, puts_raw = chain.calls, chain.puts
            atm_idx = (calls_raw["strike"] - spot).abs().argsort()[:2]
            atm_iv = float(calls_raw.iloc[atm_idx]["impliedVolatility"].mean())
            iv_rank = (
                np.clip((atm_iv - hv_min) / max(hv_max - hv_min, 0.05) * 100, 0, 100)
                if hv_max > hv_min
                else 50.0
            )

            # Filter liquid options chains
            liquid_puts = filter_liquid_chain(puts_raw, spot, T, "put")
            liquid_calls = filter_liquid_chain(calls_raw, spot, T, "call")

            trade = None
            # Route by price & IV Rank
            if spot <= 10.5 and not liquid_puts.empty:
                trade = find_cash_secured_put(liquid_puts, spot)

            if not trade and iv_rank >= 35.0 and not liquid_puts.empty:
                trade = find_bull_put_credit_spread(liquid_puts)

            if not trade and not liquid_calls.empty:
                trade = find_bull_call_debit_spread(liquid_calls)

            if trade:
                results.append({
                    "Ticker": symbol,
                    "Spot": f"${spot:.2f}",
                    "Expiry (DTE)": f"{exp_str} ({dte}d)",
                    "ATM IV (IVR)": f"{atm_iv*100:.1f}% ({iv_rank:.0f}%)",
                    **trade,
                })
        except Exception as e:
            continue

    df_out = pd.DataFrame(results)
    if df_out.empty:
        print("No setups matched all strict liquidity and pricing criteria right now.")
        print("Tip: Run during market hours (10:15 AM - 3:45 PM ET) when live bid-ask quotes are tightest.")
    else:
        print(df_out.to_string(index=False))
        df_out.to_csv("weekly_options_watchlist.csv", index=False)
        print("\nSaved results to 'weekly_options_watchlist.csv'.")


if __name__ == "__main__":
    run_weekly_scanner()

Scanning 15 tickers for weekly setups under $1,000...

Ticker    Spot    Expiry (DTE) ATM IV (IVR)                 Strategy                  Setup      Deltas Credit/Debit Collateral/Cost Max Return %
  IBIT  $45.96 2026-10-30 (5d)  37.5% (29%)   Bull Call Debit Spread Buy $45.5C / Sell $48C 0.60 / 0.15   -$95 Debit             $95       163.2%
  NVDA $210.96 2026-10-30 (5d)  42.3% (80%)   Bull Call Debit Spread Buy $210C / Sell $215C 0.55 / 0.36  -$237 Debit            $237       111.0%
  PLTR $196.75 2026-10-30 (5d)  53.3% (35%)   Bull Call Debit Spread Buy $195C / Sell $200C 0.57 / 0.41  -$250 Debit            $250       100.0%
  SOFI  $14.61 2026-10-30 (5d) 95.3% (100%)   Bull Call Debit Spread Buy $14.5C / Sell $17C 0.55 / 0.09   -$68 Debit             $68       267.6%
  SNAP   $6.34 2026-10-30 (5d)  65.6% (59%) Cash-Secured Put (Wheel)            Sell 1x $6P        0.23   +$9 Credit            $591    1.5% / wk

Saved results to 'weekly_options_watchlist.csv'.
